# 1. IEEE 57 + IEEE 14 with two HVDC grids: pandapower does not converge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WillemLam/unified-acdc-powerflow/blob/main/benchmarks/pandapower_acdc/01_ieee57_14_two_hvdc_grids.ipynb)

Two non-synchronous AC grids, the IEEE 57-bus and the IEEE 14-bus system, exchange power through two meshed
multi-terminal HVDC grids with 8 converters. This is the multi-terminal HVDC case of W. Lambrichts and M. Paolone, "General and Unified Model of the Power Flow Problem in Multiterminal AC/DC Networks", IEEE Trans. Power Systems, 2024, [doi:10.1109/TPWRS.2024.3378926](https://doi.org/10.1109/TPWRS.2024.3378926) (section IV-B, Fig. 6).

[unified-acdc-powerflow](https://github.com/WillemLam/unified-acdc-powerflow) solves it in a few iterations. pandapower 3.5.5's AC/DC power flow does not
converge. Step by step we show why: the case combines two situations on which pandapower 3.5.5 fails.

![Grid topology](https://raw.githubusercontent.com/WillemLam/unified-acdc-powerflow/main/validation/figures/ieee57_14_hvdc.png)

In [1]:
# Setup. In Colab: download the repository and install pandapower 3.5.5.
# Locally: use the repository this notebook is in.
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("unified-acdc-powerflow").exists():
        !git clone -q https://github.com/WillemLam/unified-acdc-powerflow.git
    %cd unified-acdc-powerflow
    !pip install -q "pandapower==3.5.5" pyyaml
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists():
        root = root.parent
    os.chdir(root)
sys.path.insert(0, os.getcwd())

import logging, time, warnings
logging.getLogger("pandapower").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import pandapower as pp
from unified_acdc_powerflow import run_pf
from unified_acdc_powerflow.cases import load_case

def run_pandapower(net, init="auto"):
    """pandapower's own AC/DC power flow: a status line with the iterations and time reported by pandapower."""
    t0 = time.perf_counter()
    try:
        pp.runpp(net, init=init, max_iteration=100, tolerance_mva=1e-8)
    except Exception as err:
        return f"{type(err).__name__}: {str(err)[:70]} | time: {(time.perf_counter() - t0) * 1e3:.1f} ms"
    return f"converged | iterations: {net._ppc['iterations']} | time: {net._ppc['et'] * 1e3:.1f} ms"

def report(res):
    """The same status line for unified-acdc-powerflow."""
    return (f"{res.message} | iterations: {res.iterations} | max mismatch: {res.max_mismatch:.2e} | "
            f"time: {res.solve_time_s * 1e3:.1f} ms")

def with_coupling_impedance(net, z):
    """A copy of net for pandapower: every converter behind a coupling impedance of z p.u. (AC: r = x; DC: r)."""
    n = pp.from_json_string(pp.to_json(net))
    for i, v in n.vsc.iterrows():
        z_ac = z * n.bus.vn_kv[v.bus] ** 2 / n.sn_mva
        z_dc = z * n.bus_dc.vn_kv[v.bus_dc] ** 2 / n.sn_mva
        n.vsc.loc[i, ["r_ohm", "x_ohm", "r_dc_ohm"]] = [z_ac, z_ac, z_dc]
    return n

print("pandapower", pp.__version__)


pandapower 3.5.5


## Step 1: the case

The case file `validation/cases/ieee57_14_hvdc.yaml` holds:
- all data of both IEEE systems (lines, transformers with taps, shunts, generators);
- two DC grids with one DC load;
- 8 converters, 3 of them holding a DC voltage and 5 with fixed P and Q.

The converters are connected directly to the grid buses.

In [2]:
net = load_case("ieee57_14_hvdc")
print(f"{len(net.bus)} AC buses, {len(net.bus_dc)} DC buses, {len(net.vsc)} converters, {len(net.load_dc)} DC load")
print("converters on a bus with a voltage-controlling generator:", sorted(set(net.vsc.bus) & set(net.gen.bus)))
net.vsc[["name", "bus", "bus_dc", "mode", "p_ac_mw", "q_ac_mvar", "vm_dc_pu"]]

72 AC buses, 10 DC buses, 8 converters, 1 DC load
converters on a bus with a voltage-controlling generator: [202]


,name,bus,bus_dc,mode,p_ac_mw,q_ac_mvar,vm_dc_pu
0,IC 212-1,212,1,VdcQ,0.0,-5.0,1.0
1,IC 150-2,150,2,PQ,20.0,21.0,1.0
2,IC 151-3,151,3,PQ,10.0,-31.0,1.0
3,IC 158-4,158,4,VdcQ,0.0,11.0,1.0
4,IC 202-5,202,5,VdcQ,0.0,-15.0,1.0
5,IC 205-6,205,6,PQ,45.0,30.0,1.0
6,IC 115-7,115,7,PQ,15.0,-40.0,1.0
7,IC 117-8,117,8,PQ,15.0,50.0,1.0


## Step 2: unified-acdc-powerflow

In [3]:
res = run_pf(net)
print("unified-acdc-powerflow:", report(res))
res.converter_table()

unified-acdc-powerflow: converged | iterations: 5 | max mismatch: 5.66e-14 | time: 188.4 ms


,vsc,name,mode,P_ac_pu,Q_ac_pu,P_dc_pu,loss_pu,I_pu
0,0,IC 212-1,VdcQ,-0.300602,-0.05,0.300602,0.0,0.298649
1,1,IC 150-2,PQ,0.200000,0.21,-0.200000,0.0,0.276926
2,2,IC 151-3,PQ,0.100000,-0.31,-0.100000,0.0,0.314100
3,3,IC 158-4,VdcQ,-0.294227,0.11,0.294227,0.0,0.310417
4,4,IC 202-5,VdcQ,-0.559807,-0.15,0.559807,0.0,0.554598
5,5,IC 205-6,PQ,0.450000,0.30,-0.450000,0.0,0.523604
6,6,IC 115-7,PQ,0.150000,-0.40,-0.150000,0.0,0.435832
7,7,IC 117-8,PQ,0.150000,0.50,-0.150000,0.0,0.494718


## Step 3: pandapower

pandapower models every converter behind a coupling impedance. A direct connection is approximated with a small
one, which is the same as adding a dummy bus with a tiny impedance; with exactly zero it stops with a division by
zero. We use 1e-3 p.u. of the bus base.

In [4]:
print("pandapower, zero coupling impedance:", run_pandapower(with_coupling_impedance(net, 0.0)))
print("pandapower, coupling impedance 1e-3 p.u.:", run_pandapower(with_coupling_impedance(net, 1e-3)))

pandapower, zero coupling impedance: FloatingPointError: divide by zero encountered in divide | time: 177.5 ms


pandapower, coupling impedance 1e-3 p.u.: LoadflowNotConverged: Power Flow nr did not converge after 100 iterations! | time: 1881.0 ms


## Step 4: first reason, the DC load

pandapower 3.5.5 multiplies DC loads (`load_dc`) by the MVA base `sn_mva`, here 100: the 10 MW DC load counts as
1000 MW. We cancel that in pandapower's copy of the net by dividing the DC load by `sn_mva`.

In [5]:
pp_net = with_coupling_impedance(net, 1e-3)
pp_net.load_dc["p_dc_mw"] = pp_net.load_dc.p_dc_mw / pp_net.sn_mva
print("pandapower, DC load corrected:", run_pandapower(pp_net))

pandapower, DC load corrected: LoadflowNotConverged: Power Flow nr did not converge after 100 iterations! | time: 1712.3 ms


Still no convergence.

## Step 5: second reason, a converter next to a generator

Converter `IC 202-5` sits on bus 202, whose voltage is held by a generator. This is a common situation (HVDC
stations are often built next to power plants), and pandapower 3.5.5 does not converge with it, for any coupling
impedance from 1e-4 to 1e-1 p.u. (see the benchmark README). To show that this is the second cause, we move that
converter to a new busbar next to bus 202, through a short branch (r = 0.01 p.u.), and give **both** tools this
modified grid.

In [6]:
net2 = load_case("ieee57_14_hvdc")
i = net2.vsc.index[net2.vsc.bus == 202][0]
busbar = pp.create_bus(net2, net2.bus.vn_kv[202], name="busbar of IC 202-5")
z_base = net2.bus.vn_kv[202] ** 2 / net2.sn_mva
pp.create_line_from_parameters(net2, 202, busbar, 1.0, 0.01 * z_base, 1e-4 * z_base, 0.0, 99.0)
net2.vsc.at[i, "bus"] = busbar

pp_net = with_coupling_impedance(net2, 1e-3)
pp_net.load_dc["p_dc_mw"] = pp_net.load_dc.p_dc_mw / pp_net.sn_mva      # DC-load correction of step 4
print("pandapower:", run_pandapower(pp_net))
res2 = run_pf(net2)
print("unified-acdc-powerflow:", report(res2))

m = res2.model
dv_ac = max(abs(pp_net.res_bus.vm_pu[b] - abs(res2.E_ac[m.bus_lookup[b]])) for b in net2.bus.index)
dv_dc = max(abs(pp_net.res_bus_dc.vm_pu[b] - res2.E_dc[m.bus_dc_lookup[b]]) for b in net2.bus_dc.index)
print(f"largest voltage difference: AC {dv_ac:.1e} p.u., DC {dv_dc:.1e} p.u.")

pandapower: converged | iterations: 4 | time: 84.3 ms


unified-acdc-powerflow: converged | iterations: 5 | max mismatch: 5.66e-14 | time: 234.8 ms
largest voltage difference: AC 1.9e-05 p.u., DC 6.2e-14 p.u.


Now pandapower converges, and the two tools agree. The remaining difference comes from the coupling
impedance in pandapower's model; notebook 2 shows against EMTP-RV that it is not needed.

**Takeaway.** On this multi-terminal HVDC case, pandapower 3.5.5 does not converge because:
- it multiplies DC loads by `sn_mva`;
- it does not converge with a converter on the bus of a voltage-controlling generator.

unified-acdc-powerflow solves the original case directly, with the converters connected to their buses. Where
pandapower converges, the two agree up to the effect of its coupling impedance.